# LangChain Expression Language (LCEL)

In [1]:
import os
import openai

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file
openai.api_key = os.environ['OPENAI_API_KEY']

In [26]:
llm_model = os.getenv("OPENAI_MODEL", "gpt-4o")

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

## Simple Chain

In [3]:
prompt = ChatPromptTemplate.from_template(
    "tell me a short joke about {topic}"
)
model = ChatOpenAI()
output_parser = StrOutputParser()

In [4]:
chain = prompt | model | output_parser

In [5]:
chain.invoke({"topic": "bears"})

'Why did the bear bring a flashlight into the cave? \nBecause it wanted to see if there were any bear-y scary monsters inside!'

## More complex chain

And Runnable Map to supply user-provided inputs to the prompt.

In [6]:
from langchain_openai import OpenAIEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

In [7]:
vectorstore = InMemoryVectorStore.from_texts(
    ["harrison worked at kensho", "bears like to eat honey"],
    embedding=OpenAIEmbeddings()
)
retriever = vectorstore.as_retriever()

In [10]:
retriever.invoke("where did harrison work?")

[Document(id='7368d7cf-340e-422a-92a4-f548a95c1e29', metadata={}, page_content='harrison worked at kensho'),
 Document(id='03e14b42-cf88-423c-a8f7-280430eac090', metadata={}, page_content='bears like to eat honey')]

In [11]:
retriever.invoke("what do bears like to eat")

[Document(id='03e14b42-cf88-423c-a8f7-280430eac090', metadata={}, page_content='bears like to eat honey'),
 Document(id='7368d7cf-340e-422a-92a4-f548a95c1e29', metadata={}, page_content='harrison worked at kensho')]

In [12]:
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

In [13]:
from langchain_core.runnables import RunnableParallel

In [14]:
chain = RunnableParallel({
    "context": lambda x: retriever.invoke(x["question"]),
    "question": lambda x: x["question"]
}) | prompt | model | output_parser

In [15]:
chain.invoke({"question": "where did harrison work?"})

'Harrison worked at Kensho.'

In [16]:
inputs = RunnableParallel({
    "context": lambda x: retriever.invoke(x["question"]),
    "question": lambda x: x["question"]
})

In [17]:
inputs.invoke({"question": "where did harrison work?"})

{'context': [Document(id='7368d7cf-340e-422a-92a4-f548a95c1e29', metadata={}, page_content='harrison worked at kensho'),
  Document(id='03e14b42-cf88-423c-a8f7-280430eac090', metadata={}, page_content='bears like to eat honey')],
 'question': 'where did harrison work?'}

## Bind

and OpenAI Functions

In [18]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "weather_search",
            "description": "Search for weather given an airport code",
            "parameters": {
                "type": "object",
                "properties": {
                    "airport_code": {
                        "type": "string",
                        "description": "The airport code to get the weather for"
                    },
                },
                "required": ["airport_code"]
            }
        }
    }
]

In [27]:
prompt = ChatPromptTemplate.from_messages(
    [
        ("human", "{input}")
    ]
)
model = ChatOpenAI(temperature=0, model=llm_model).bind_tools(tools)

In [28]:
runnable = prompt | model

In [29]:
runnable.invoke({"input": "what is the weather in sf"})

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 93, 'total_tokens': 109, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_2ca5b70601', 'id': 'chatcmpl-DSaqt98AvvIF3bx128QERWt5vD7XK', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d7065-d015-7230-9507-c5b56fe7d726-0', tool_calls=[{'name': 'weather_search', 'args': {'airport_code': 'SFO'}, 'id': 'call_DGCUBBwt5zjLSqGTlbAnB4Xr', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 93, 'output_tokens': 16, 'total_tokens': 109, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [30]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "weather_search",
            "description": "Search for weather given an airport code",
            "parameters": {
                "type": "object",
                "properties": {
                    "airport_code": {
                        "type": "string",
                        "description": "The airport code to get the weather for"
                    },
                },
                "required": ["airport_code"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "sports_search",
            "description": "Search for news of recent sport events",
            "parameters": {
                "type": "object",
                "properties": {
                    "team_name": {
                        "type": "string",
                        "description": "The sports team to search for"
                    },
                },
                "required": ["team_name"]
            }
        }
    }
]

In [31]:
model = model.bind_tools(tools)

In [32]:
runnable = prompt | model

In [33]:
runnable.invoke({"input": "how did the patriots do yesterday?"})

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 18, 'prompt_tokens': 95, 'total_tokens': 113, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-2024-08-06', 'system_fingerprint': 'fp_2ca5b70601', 'id': 'chatcmpl-DSasHrI3mZGuJk4ZsFto2AEHDIEEm', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d7067-1db7-7991-bcfa-b86f95d8edba-0', tool_calls=[{'name': 'sports_search', 'args': {'team_name': 'Patriots'}, 'id': 'call_TK5TX9riBiJAGVQjaTi1aRiS', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 95, 'output_tokens': 18, 'total_tokens': 113, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

## Fallbacks

In [34]:
from langchain_openai import OpenAI
import json

In [39]:
simple_model = OpenAI(
    temperature=0,
    max_tokens=1000,
    model="gpt-3.5-turbo-instruct"
)
simple_chain = simple_model | json.loads

In [40]:
challenge = "write three poems in a json blob, where each poem is a json blob of a title, author, and first line"

In [41]:
simple_model.invoke(challenge)

'\n\n{\n    "title": "Autumn Leaves",\n    "author": "Emily Dickinson",\n    "first_line": "The leaves are falling, one by one"\n}\n\n{\n    "title": "The Ocean\'s Song",\n    "author": "Pablo Neruda",\n    "first_line": "I hear the ocean\'s song, a symphony of waves"\n}\n\n{\n    "title": "A Winter\'s Night",\n    "author": "Robert Frost",\n    "first_line": "The snow falls softly, covering the ground"\n}'

<p style=\"background-color:#F5C780; padding:15px\"><b>Note:</b> The next line is expected to fail.</p>

In [42]:
simple_chain.invoke(challenge)

JSONDecodeError: Extra data: line 9 column 1 (char 125)

In [43]:
model = ChatOpenAI(temperature=0)
chain = model | StrOutputParser() | json.loads

In [44]:
chain.invoke(challenge)

{'poem1': {'title': 'The Rose',
  'author': 'Emily Dickinson',
  'firstLine': 'A rose by any other name would smell as sweet'},
 'poem2': {'title': 'The Road Not Taken',
  'author': 'Robert Frost',
  'firstLine': 'Two roads diverged in a yellow wood'},
 'poem3': {'title': 'Hope is the Thing with Feathers',
  'author': 'Emily Dickinson',
  'firstLine': 'Hope is the thing with feathers that perches in the soul'}}

In [45]:
final_chain = simple_chain.with_fallbacks([chain])

In [46]:
final_chain.invoke(challenge)

{'poem1': {'title': 'The Rose',
  'author': 'Emily Dickinson',
  'firstLine': 'A rose by any other name would smell as sweet'},
 'poem2': {'title': 'The Road Not Taken',
  'author': 'Robert Frost',
  'firstLine': 'Two roads diverged in a yellow wood'},
 'poem3': {'title': 'Hope is the Thing with Feathers',
  'author': 'Emily Dickinson',
  'firstLine': 'Hope is the thing with feathers that perches in the soul'}}

## Interface

In [47]:
prompt = ChatPromptTemplate.from_template(
    "Tell me a short joke about {topic}"
)
model = ChatOpenAI()
output_parser = StrOutputParser()

chain = prompt | model | output_parser

In [48]:
chain.invoke({"topic": "bears"})

'Why do bears have hairy coats?\n \nFur warmth!'

In [49]:
chain.batch([{"topic": "bears"}, {"topic": "frogs"}])

['Why do bears have hairy coats?\n-\nFur protection.',
 'What do you call a frog with no hind legs?\n\nUnhoppy!']

In [50]:
for t in chain.stream({"topic": "bears"}):
    print(t)


Why
 don
’t
 bears
 wear
 shoes
?
 


Because
 they
 have
 bear
 feet
!





In [51]:
response = await chain.ainvoke({"topic": "bears"})
response

'Why did the bear dissolve in water? Because it was polar!'